# ERA5-Land Daily Maximum Temperature Download — Ouagadougou


**Dataset:** [ERA5-Land Daily Statistics](https://cds.climate.copernicus.eu/datasets/derived-era5-land-daily-statistics)  
**Variable:** 2-metre daily maximum air temperature (`t2m`)  
**Coverage:** 2001–2020 (baseline) + 2024 (target year)  
**Spatial domain:** Ouagadougou, Burkina Faso (~0.1° ERA5-Land grid)

---

## Overview

This notebook automates the retrieval and preparation of ERA5-Land gridded temperature data for urban heat analysis over Ouagadougou. The workflow is structured in five steps:

1. **Environment setup** — install and import all required libraries  
2. **Study area definition** — derive the bounding box from a local shapefile  
3. **Interactive preview** — visualise the ERA5 bounding box on a slippy map  
4. **Data download** — retrieve yearly `.nc` files from the Copernicus CDS  
5. **Post-processing** — merge files, validate the grid, and plot a quick overview

> **Prerequisites**  
> - A valid [Copernicus CDS](https://cds.climate.copernicus.eu/) account  
> - A `.cdsapirc` credentials file in your home directory (see [CDS How-To](https://cds.climate.copernicus.eu/how-to-api))  
> - The Ouagadougou shapefile at `../../data/raw/Shapefile Ouaga/Ouaga.shp`

---
## 1 · Environment Setup

Install all dependencies and import the libraries used throughout the notebook.  
If you are working inside a dedicated conda/virtual environment, the `%pip install` calls can be skipped after the first run.

In [ ]:
# ── Install dependencies (comment out after first run) ──────────────────────
%pip install -q geopandas shapely pyproj folium
%pip install -q cdsapi
%pip install -q netCDF4
%pip install -q xarray cartopy matplotlib

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import glob

# ── Scientific stack ────────────────────────────────────────────────────────
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle

# ── Mapping ──────────────────────────────────────────────────────────────────
import folium
from shapely.geometry import box
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ── CDS API ──────────────────────────────────────────────────────────────────
import cdsapi

print("All libraries imported successfully.")

---
## 2 · Study Area Definition

We load the Ouagadougou administrative boundary (WGS-84 / EPSG:4326) and derive the axis-aligned bounding box required by the CDS API.  
The ERA5-Land API expects the area in the order **[North, West, South, East]**.

In [ ]:
# ── Load and reproject shapefile ─────────────────────────────────────────────
SHP_PATH = "../../data/raw/Shapefile Ouaga/Ouaga.shp"

gdf = gpd.read_file(SHP_PATH).to_crs(epsg=4326)
print(f"Shapefile loaded  |  CRS: {gdf.crs}  |  Features: {len(gdf)}")

# ── Compute bounding box ─────────────────────────────────────────────────────
minx, miny, maxx, maxy = gdf.total_bounds

# ERA5 format: [North, West, South, East]
# Slightly enlarged to ensure full grid-cell coverage
BBOX = [12.5, -1.7, 12.3, -1.4]   # [N, W, S, E]  ← refined after visual check

print(f"Shapefile bounds  |  lon [{minx:.4f}, {maxx:.4f}]  lat [{miny:.4f}, {maxy:.4f}]")
print(f"ERA5 request bbox |  {BBOX}  (N, W, S, E)")

---
## 3 · Interactive Map Preview

A quick sanity check: the city boundary (blue) and the ERA5 request area (red) are overlaid on a slippy map.  
The map is also saved as `ouagadougou_bbox_map.html` for external review.

In [ ]:
north, west, south, east = BBOX

# ── Build bbox polygon ───────────────────────────────────────────────────────
bbox_polygon = box(west, south, east, north)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_polygon], crs="EPSG:4326")

# ── Base map ─────────────────────────────────────────────────────────────────
map_center = [(north + south) / 2, (west + east) / 2]
m = folium.Map(location=map_center, zoom_start=12, tiles="OpenStreetMap")

# City boundary — blue
folium.GeoJson(
    gdf[["geometry"]],
    name="Ouagadougou boundary",
    style_function=lambda _: {"color": "steelblue", "weight": 2.5, "fillOpacity": 0},
    tooltip="Ouagadougou",
).add_to(m)

# ERA5 bounding box — red
folium.GeoJson(
    bbox_gdf,
    name="ERA5-Land request area",
    style_function=lambda _: {"color": "crimson", "weight": 2, "fillOpacity": 0.15},
    tooltip="ERA5-Land bbox",
).add_to(m)

folium.LayerControl().add_to(m)
m.save("ouagadougou_bbox_map.html")
print("Map saved → ouagadougou_bbox_map.html")
m   # renders inline in Jupyter

---
## 4 · ERA5-Land Data Download

We retrieve **daily maximum 2-metre temperature** for each year separately.  
Splitting by year keeps individual files manageable and allows resuming a failed run without re-downloading completed years.

| Parameter | Value |
|---|---|
| Dataset | `derived-era5-land-daily-statistics` |
| Variable | `2m_temperature` |
| Statistic | `daily_maximum` |
| Time zone | `UTC+00:00` |
| Baseline period | 2001–2020 |
| Target year | 2024 |
| Spatial domain | 12.3°N – 12.5°N, 1.7°W – 1.4°W |

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATADIR = "./"                                        # output directory
os.makedirs(DATADIR, exist_ok=True)

BASELINE_YEARS = list(map(str, range(2001, 2021)))    # 2001–2020
TARGET_YEARS   = ["2024"]                             # analysis year
ALL_YEARS      = BASELINE_YEARS + TARGET_YEARS

# ── Initialise CDS client ─────────────────────────────────────────────────────
# Credentials are read from ~/.cdsapirc  (set CDSAPI_RC to override the path)
client = cdsapi.Client()
print(f"CDS client ready  |  Years to download: {len(ALL_YEARS)}  ({ALL_YEARS[0]}–{ALL_YEARS[-1]})")

In [ ]:
# ── Download loop — one .zip per year ────────────────────────────────────────
#
# The CDS delivers NetCDF data inside a .zip archive.
# After download, rename the archive to .nc:
#   bash:  for f in *.zip; do mv "$f" "${f%.zip}.nc"; done
#

for year in ALL_YEARS:
    out_path = os.path.join(DATADIR, f"Era5_land_{year}_Ouagadougou_daily.zip")

    if os.path.exists(out_path):
        print(f"[SKIP]  {year} — file already exists")
        continue

    request = {
        "variable":        ["2m_temperature"],
        "year":            year,
        "month":           [f"{m:02d}" for m in range(1, 13)],
        "day":             [f"{d:02d}" for d in range(1, 32)],
        "daily_statistic": "daily_maximum",
        "time_zone":       "utc+00:00",
        "frequency":       "1_hourly",
        "area":            BBOX,  # [N, W, S, E]
    }

    try:
        client.retrieve("derived-era5-land-daily-statistics", request).download(out_path)
        print(f"[OK]    {year} → {out_path}")
    except Exception as err:
        print(f"[FAIL]  {year} — {err}")

### 4.1 · Rename Archives to NetCDF

The Copernicus CDS wraps the NetCDF file in a `.zip` container.  
Run the cell below (or the equivalent shell one-liner) to strip the `.zip` extension so xarray can open the files directly.

```bash
# One-liner — run in the data directory
for f in *.zip; do mv "$f" "${f%.zip}.nc"; done
```

In [ ]:
# ── Rename .zip → .nc (Python equivalent) ────────────────────────────────────
renamed = 0
for fpath in glob.glob(os.path.join(DATADIR, "Era5_land_*_Ouagadougou_daily.zip")):
    new_path = fpath.replace(".zip", ".nc")
    os.rename(fpath, new_path)
    renamed += 1

print(f"Renamed {renamed} file(s)  (.zip → .nc)")

---
## 5 · Post-Processing

### 5.1 · Merge Yearly Files into a Single Dataset

In [ ]:
# ── Find all annual files ─────────────────────────────────────────────────────
nc_files = sorted(glob.glob(os.path.join(DATADIR, "Era5_land_*_Ouagadougou_daily.nc")))
print(f"Found {len(nc_files)} annual file(s):")
for f in nc_files:
    print(f"  {os.path.basename(f)}")

# ── Concatenate along the time dimension ─────────────────────────────────────
MERGED_FILE = os.path.join(DATADIR, "Ouagadougou_2001_2024_daily_tmax.nc")

if not os.path.exists(MERGED_FILE):
    datasets = [xr.open_dataset(f) for f in nc_files]
    ds_merged = xr.concat(datasets, dim="time")
    ds_merged.to_netcdf(MERGED_FILE)
    print(f"\nMerged dataset written → {MERGED_FILE}")
else:
    ds_merged = xr.open_dataset(MERGED_FILE)
    print(f"\nLoaded existing merged file → {MERGED_FILE}")

print(ds_merged)

### 5.2 · Grid Validation — ERA5-Land Points over Ouagadougou

The plot below confirms which ERA5-Land grid cells cover the study area.  
Each red square represents one ~0.1° × 0.1° grid cell; its centre point is marked by a red dot.

In [ ]:
# ── Extract grid metadata ─────────────────────────────────────────────────────
era_lats = ds_merged.latitude.values
era_lons = ds_merged.longitude.values
grid_res = float(np.abs(np.diff(era_lats)).mean())

print(f"ERA5-Land latitudes  : {era_lats}")
print(f"ERA5-Land longitudes : {era_lons}")
print(f"Grid resolution      : {grid_res:.4f}°")

In [ ]:
# ── Visualise grid cells over the shapefile ───────────────────────────────────
minx, miny, maxx, maxy = gdf.total_bounds
padding = 0.08

fig, ax = plt.subplots(figsize=(11, 9))

# Shapefile fill
gdf.plot(ax=ax, facecolor="#cce5ff", edgecolor="#003f7f", linewidth=2.5, alpha=0.7)

# ERA5 grid cells
for lat in era_lats:
    for lon in era_lons:
        rect = Rectangle(
            (lon - grid_res / 2, lat - grid_res / 2),
            grid_res, grid_res,
            linewidth=1.5, edgecolor="#cc0000",
            facecolor="#ff4444", alpha=0.12, linestyle="--",
        )
        ax.add_patch(rect)
        ax.plot(lon, lat, "o", color="#cc0000", markersize=8, zorder=5)
        ax.text(
            lon, lat + 0.012,
            f"({lon:.2f}°, {lat:.2f}°)",
            fontsize=7.5, ha="center",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="#fff9c4", alpha=0.85),
        )

# Shapefile bounding box
ax.plot(
    [minx, maxx, maxx, minx, minx],
    [miny, miny, maxy, maxy, miny],
    "g--", linewidth=2, label="Shapefile extent",
)

# Formatting
ax.set_xlim(era_lons.min() - padding, era_lons.max() + padding)
ax.set_ylim(era_lats.min() - padding, era_lats.max() + padding)
ax.set_aspect("equal")
ax.set_xlabel("Longitude (°E)", fontsize=12)
ax.set_ylabel("Latitude (°N)", fontsize=12)
ax.set_title(
    "ERA5-Land Grid Points over Ouagadougou\n"
    f"({len(era_lats)} × {len(era_lons)} cells, resolution ≈ {grid_res:.3f}°)",
    fontsize=13, fontweight="bold",
)
ax.grid(True, alpha=0.3, linestyle=":")

# Legend / info box
legend_handles = [
    mpatches.Patch(facecolor="#cce5ff", edgecolor="#003f7f", label="Ouagadougou boundary"),
    mpatches.Patch(facecolor="#ff4444", edgecolor="#cc0000", alpha=0.4, label="ERA5-Land grid cell"),
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=10)

info = (
    f"Grid: {len(era_lats)} lat × {len(era_lons)} lon\n"
    f"Resolution: {grid_res:.3f}°\n"
    f"Period: 2001–2024"
)
ax.text(
    0.02, 0.98, info, transform=ax.transAxes,
    fontsize=10, va="top",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.9),
)

plt.tight_layout()
plt.savefig("era5_grid_ouagadougou.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → era5_grid_ouagadougou.png")

### 5.3 · Quick Time-Series Overview

Spatial average of daily maximum temperature across all ERA5-Land grid cells covering Ouagadougou, plotted for the full merged period.

In [ ]:
# ── Detect temperature variable name ─────────────────────────────────────────
t_var = [v for v in ds_merged.data_vars if "t2m" in v or "temperature" in v][0]
print(f"Temperature variable: '{t_var}'")

# ── Spatial mean (all grid cells) → convert K to °C ──────────────────────────
t_spatial_mean = ds_merged[t_var].mean(dim=["latitude", "longitude"]) - 273.15

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

t_spatial_mean.plot(
    ax=ax, color="#cc2222", linewidth=0.6, alpha=0.75, label="Daily Tmax"
)

# Annual rolling mean
t_rolling = t_spatial_mean.rolling(time=365, center=True, min_periods=30).mean()
t_rolling.plot(ax=ax, color="#000000", linewidth=2, label="365-day rolling mean")

ax.set_title(
    "ERA5-Land Daily Maximum 2-m Temperature — Ouagadougou (spatial mean)",
    fontsize=12, fontweight="bold",
)
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, linestyle=":")

plt.tight_layout()
plt.savefig("era5_tmax_timeseries_ouagadougou.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → era5_tmax_timeseries_ouagadougou.png")

---
## Summary

| Output file | Description |
|---|---|
| `ouagadougou_bbox_map.html` | Interactive map of the ERA5 request area |
| `Era5_land_<year>_Ouagadougou_daily.nc` | Annual raw ERA5-Land files |
| `Ouagadougou_2001_2024_daily_tmax.nc` | **Merged analysis-ready dataset** |
| `era5_grid_ouagadougou.png` | Grid cell validation figure |
| `era5_tmax_timeseries_ouagadougou.png` | Temperature time-series overview |

The merged dataset `Ouagadougou_2001_2024_daily_tmax.nc` is ready for downstream analysis (e.g. heat-wave detection, urban heat island assessment, bias correction).

---

